<a href="https://colab.research.google.com/github/Genjuzu/Computer-Schule/blob/main/pii_modell_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Eigenes deutsches PII-Modell trainieren (Google Colab)

Dieses Notizbuch trainiert **zwei** kleine Modelle nacheinander, die persönliche Daten in deutschen Texten finden
(Namen, Adressen, Telefonnummern, Gesundheitsangaben und so weiter):

1. **mmBERT-small** – der Hauptkandidat (MIT-Lizenz).
2. **distilbert-base-german-cased** – Plan B mit sicherer Lizenz (Apache-2.0, eigener deutscher Tokenizer).

**So gehst du vor:**
1. Oben im Menü: *Laufzeit → Laufzeittyp ändern → T4-GPU* (oder L4, falls verfügbar).
2. Dann *Laufzeit → Alle ausführen*.
3. Wenn danach gefragt wird: die Datei `trainingsdaten.jsonl.gz` hochladen.
4. Warten (etwa 1,5 bis 2 Stunden). Am Ende lädt der Browser **zwei** Zip-Dateien herunter
   (der Browser fragt eventuell, ob mehrere Downloads erlaubt sind – bitte erlauben).
5. Beide Zip-Dateien zurückgeben. Sie werden dann auf dem versteckten Prüfsatz gemessen.

Du brauchst **kein** Passwort und **keinen** Zugangsschlüssel. Beide Basismodelle sind frei verfügbar.

## 1. Stellschrauben

Hier stehen alle Einstellungen. Die Werte sind so gewählt, dass **ein** Lauf reicht
(Startwerte aus Bericht 17). Du musst normalerweise nichts ändern.

In [6]:
import os

MODELLE = [
    # name, Basismodell, Leerzeichen vor jeden Text (Hinweis der mmBERT-Autoren), Zeitlimit in Minuten
    {"name": "mmbert-small", "basis": "jhu-clsp/mmBERT-small", "leerzeichen_vorne": True, "zeitlimit_min": 65},
    {"name": "distilbert-de", "basis": "distilbert/distilbert-base-german-cased", "leerzeichen_vorne": False, "zeitlimit_min": 40},
]
EPOCHEN = 3                 # Wie oft das Modell alle Trainingssätze sieht
LERNRATE_ENCODER = 3e-5     # Wie stark das vortrainierte Modell pro Schritt lernt
LERNRATE_KOPF = 1e-4        # Lernrate für den neuen Klassifikations-Kopf (lernt schneller)
WARMUP = 0.06               # Anteil der Schritte, in denen die Lernrate langsam steigt
BATCH = 32                  # Sätze pro Lernschritt
MAX_LAENGE = 128            # Längere Sätze werden abgeschnitten (in Wortteilen)
GEWICHT_O = 0.4             # Gewicht für "keine persönliche Angabe" (< 1 = lieber einmal zu viel schwärzen)
ZIEL_FEHLALARM = 0.05       # Höchstens 5 % harmlose Sätze mit Fehlalarm (für die Wahl der Schwelle)
PRUEFUNGEN_JE_EPOCHE = 2    # Wie oft pro Epoche geprüft wird (der beste Stand wird behalten)
KD_ALPHA, KD_TEMPERATUR = 0.5, 2.0   # Destillation – nur aktiv, wenn das Paket weiche Lehrer-Labels enthält
MAX_SAETZE = None           # None = alle Trainingssätze nehmen; sonst z. B. 50000
ONNX_PRUEF_SAETZE = 2500    # so viele Validierungssätze prüfen die ONNX-Dateien nach dem Export (CPU)
LEHRER_SAETZE_NUTZEN = True # Sätze, die das GLiNER-Lehrermodell markiert hat
DATEI = "trainingsdaten.jsonl.gz"
IN_DRIVE_SPEICHERN = False  # True = Zwischenstände und Zips zusätzlich in Google Drive
DRIVE_ORDNER = "/content/drive/MyDrive/pii-modell"
SAAT = 42

# Nur für den Probelauf auf einem normalen Rechner (ohne GPU). In Colab bleibt das aus.
PROBE = os.environ.get("PII_PROBELAUF") == "1"
MAX_SCHRITTE = None
if PROBE:
    EPOCHEN, MAX_SAETZE, BATCH, MAX_SCHRITTE = 1, 32, 4, 1
    DATEI = os.environ.get("PII_DATEI", DATEI)
print("Probelauf" if PROBE else "Volles Training", "| Modelle:", [m["name"] for m in MODELLE], "| Epochen:", EPOCHEN, "| Lernrate Encoder/Kopf:", LERNRATE_ENCODER, "/", LERNRATE_KOPF)

Volles Training | Modelle: ['mmbert-small', 'distilbert-de'] | Epochen: 3 | Lernrate Encoder/Kopf: 3e-05 / 0.0001


## 2. Ist eine GPU da?

In [7]:
import torch
if torch.cuda.is_available():
    GERAET = "cuda"
    faehigkeit = torch.cuda.get_device_capability(0)
    # L4/A100 (ab Ampere): bf16, stabiler. T4 (Turing): fp16 mit GradScaler.
    GENAUIGKEIT = "bf16" if faehigkeit >= (8, 0) and torch.cuda.is_bf16_supported() else "fp16"
    print("GPU gefunden:", torch.cuda.get_device_name(0), "| Rechenformat:", GENAUIGKEIT)
else:
    GERAET, GENAUIGKEIT = "cpu", "fp32"
    if PROBE:
        print("Keine GPU – Probelauf auf der CPU.")
    else:
        raise SystemExit("Keine GPU gefunden! Bitte oben: Laufzeit → Laufzeittyp ändern → T4-GPU, dann neu starten.")
AUTOCAST = {"bf16": torch.bfloat16, "fp16": torch.float16}.get(GENAUIGKEIT)

GPU gefunden: Tesla T4 | Rechenformat: fp16


## 3. Programme installieren

Die Versionen sind fest eingestellt. Genau mit diesen Versionen wurde das Notizbuch geprüft
(eine andere transformers-Version kann den Export unbemerkt verändern).

In [8]:
import sys, subprocess
PAKETE = ["transformers==5.16.1", "onnx==1.23.2", "onnxruntime==1.31.0", "onnxscript==0.7.2", "pandas"]
if not PROBE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + PAKETE, check=True)
import transformers, onnx, onnxruntime
print("transformers", transformers.__version__, "| onnx", onnx.__version__, "| onnxruntime", onnxruntime.__version__, "| torch", torch.__version__)

transformers 5.16.1 | onnx 1.23.2 | onnxruntime 1.31.0 | torch 2.11.0+cu130


## 4. Trainingsdaten hochladen

Es erscheint ein Knopf **„Dateien auswählen“**. Dort `trainingsdaten.jsonl.gz` wählen.
Liegt die Datei schon in Google Drive (Ordner `pii-modell`), wird sie von dort genommen.

In [5]:
import os, shutil
IN_COLAB = False
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB and IN_DRIVE_SPEICHERN:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_ORDNER, exist_ok=True)
    if not os.path.exists(DATEI) and os.path.exists(os.path.join(DRIVE_ORDNER, DATEI)):
        shutil.copy(os.path.join(DRIVE_ORDNER, DATEI), DATEI)

if not os.path.exists(DATEI):
    if IN_COLAB:
        from google.colab import files
        hoch = files.upload()
        name = next(iter(hoch))
        if name != DATEI:
            os.rename(name, DATEI)
    else:
        raise SystemExit("Datei nicht gefunden: " + DATEI)
print("Daten:", DATEI, round(os.path.getsize(DATEI) / 1e6, 1), "MB")

Saving trainingsdaten.jsonl.gz to trainingsdaten.jsonl.gz
Daten: trainingsdaten.jsonl.gz 11.6 MB


## 5. Daten einlesen

In [9]:
import gzip, json, random, collections
random.seed(SAAT)
with gzip.open(DATEI, "rt", encoding="utf-8") as f:
    meta = json.loads(f.readline())["_meta"]
    zeilen = [json.loads(z) for z in f]
LABELS = meta["labels"]            # z. B. PERSON, ADRESSE, ..., ART9, ORT, ORG
HILFS = set(meta["hilfslabels"])   # ORT, ORG: werden mitgelernt, aber nie geschwärzt
TAGS = ["O"] + [p + "-" + l for l in LABELS for p in ("B", "I")]
T2I = {t: i for i, t in enumerate(TAGS)}
SCHWAERZEN = [i for i, t in enumerate(TAGS) if t != "O" and t[2:] not in HILFS]   # Klassen, die geschwärzt werden

train = [z for z in zeilen if z["split"] == "train" and (LEHRER_SAETZE_NUTZEN or z["quelle"] != "lehrer")]
val = [z for z in zeilen if z["split"] == "val"]
random.shuffle(train)
if MAX_SAETZE:
    train = train[:MAX_SAETZE]
    val = [z for q in sorted(set(x["quelle"] for x in val)) for z in [x for x in val if x["quelle"] == q][:6]]
MIT_WEICH = any("weich" in z for z in train)
print("Training:", len(train), "Sätze | Validierung:", len(val), "Sätze | weiche Lehrer-Labels:", "ja" if MIT_WEICH else "nein (nur harte Labels)")
print("Quellen im Training:", dict(collections.Counter(z["quelle"] for z in train)))
print("Quellen in der Validierung:", dict(collections.Counter(z["quelle"] for z in val)))
print("Stil im Training:", dict(collections.Counter(z.get("stil", "text") for z in train)))

Training: 120439 Sätze | Validierung: 6866 Sätze | weiche Lehrer-Labels: nein (nur harte Labels)
Quellen im Training: {'baukasten': 44930, 'openpii': 25718, 'germeval': 25200, 'nobody': 9550, 'kiji': 11528, 'lehrer': 3513}
Quellen in der Validierung: {'dev': 404, 'baukasten': 3531, 'openpii': 1000, 'germeval': 1000, 'kiji': 488, 'nobody': 443}
Stil im Training: {'asr': 54593, 'text': 45395, 'asr_vorher': 6774, 'asr_roh': 13677}


## 6. Werkzeuge: Zerlegen, Prüfen, Schwelle

Gemessen wird wie beim Filter:
* **Entitäts-Leck** (Hauptzahl, streng auf Zeichenebene) = eine persönliche Angabe, von der auch nur **ein**
  Buchstabe oder eine Ziffer sichtbar bleibt. Ein halb geschwärzter Name zählt als Leck. Angabe: je 1000 Stellen.
* **Satz-Leck** (wie in den bisherigen Berichten) = ein Satz, in dem eine persönliche Angabe nicht (ganz) geschwärzt wurde. Je 1000 Sätze mit PII.
* **Fehlalarm** = ein harmloser Satz, in dem trotzdem etwas geschwärzt wurde.

Geschwärzt wird ein Wortteil, wenn die Wahrscheinlichkeit „persönliche Angabe“ mindestens **τ** ist.
τ wird nach dem Training auf der Validierung gewählt (nach einer Temperatur-Kalibrierung).

In [10]:
import numpy as np, pandas as pd, torch.nn.functional as F

def zerlegen(tok, texte, leer, mit_offsets=True, **kw):
    """Tokenisiert; bei leer=True wird ein Leerzeichen vorangestellt und die Offsets zurückgerechnet."""
    enc = tok([(" " + t) if leer else t for t in texte], truncation=True, max_length=MAX_LAENGE,
              return_offsets_mapping=True, **kw)
    if leer:
        neu = []
        for offs in enc["offset_mapping"]:
            o2 = []
            for a, b in (offs.tolist() if hasattr(offs, "tolist") else offs):
                a, b = max(a - 1, 0), b - 1
                o2.append((a, b) if b > a else (0, 0))
            neu.append(o2)
        enc["offset_mapping"] = neu
    return enc

def kodieren(tok, z, leer):
    enc = zerlegen(tok, [z["text"]], leer)
    offs = enc["offset_mapping"][0]
    tags = []
    for i, (a, b) in enumerate(offs):
        if a == b:
            tags.append(-100); continue
        t = 0
        for s, e, lab in z["spans"]:
            if a < e and b > s:
                vorher = offs[i - 1] if i > 0 else (0, 0)
                anfang = vorher[0] == vorher[1] or not (vorher[0] < e and vorher[1] > s)
                t = T2I[("B-" if anfang else "I-") + lab]; break
        tags.append(t)
    return {"input_ids": enc["input_ids"][0], "labels": tags}

def stapel(tok, liste):
    n = max(len(k["input_ids"]) for k in liste)
    pad = tok.pad_token_id if tok.pad_token_id is not None else 0
    ids = torch.full((len(liste), n), pad); att = torch.zeros((len(liste), n), dtype=torch.long)
    lab = torch.full((len(liste), n), -100)
    for i, k in enumerate(liste):
        L = len(k["input_ids"])
        ids[i, :L] = torch.tensor(k["input_ids"]); att[i, :L] = 1; lab[i, :L] = torch.tensor(k["labels"])
    return ids, att, lab

def logits_fuer(model, tok, saetze, leer, groesse=64):
    """Gibt je Satz (offsets, logits[numpy]) zurück."""
    model.eval(); aus = []
    with torch.no_grad():
        for i in range(0, len(saetze), groesse):
            teil = saetze[i:i + groesse]
            enc = zerlegen(tok, [z["text"] for z in teil], leer, padding=True, return_tensors="pt")
            offs = enc.pop("offset_mapping")
            with torch.autocast(GERAET, dtype=AUTOCAST, enabled=AUTOCAST is not None and GERAET == "cuda"):
                lg = model(input_ids=enc["input_ids"].to(GERAET), attention_mask=enc["attention_mask"].to(GERAET)).logits
            lg = lg.float().cpu().numpy()
            for j, m in enumerate(enc["attention_mask"].tolist()):
                L = sum(m)
                aus.append((list(offs[j])[:L], lg[j, :L]))
    return aus

def spannen(offs, logits, tau, temp=1.0):
    """Schwärzen, wenn P(schwärzen) = Summe der schwärzbaren Klassen >= tau. Label = wahrscheinlichste schwärzbare Klasse."""
    p = torch.softmax(torch.tensor(logits) / temp, -1).numpy()
    sp = []
    for (a, b), pr in zip(offs, p):
        a, b = int(a), int(b)
        if a == b:
            continue
        ps = pr[SCHWAERZEN]
        if ps.sum() < tau:
            continue
        lab = TAGS[SCHWAERZEN[int(ps.argmax())]][2:]
        if sp and sp[-1][0] == lab and a <= sp[-1][2] + 1:
            sp[-1][2] = b
        else:
            sp.append([lab, a, b])
    return [tuple(s) for s in sp]

def bewerten(saetze, vorh):
    zaehl = collections.defaultdict(lambda: [0, 0, 0])   # richtig, vorhergesagt, echt
    ent = collections.defaultdict(lambda: [0, 0])          # je Label: Stellen, davon geleakt
    lecks = pii = fa = neg = 0
    for z, v in zip(saetze, vorh):
        echt = [(l, s, e) for s, e, l in z["spans"] if l not in HILFS]
        for l, s, e in echt: zaehl[l][2] += 1
        for l, s, e in v: zaehl[l][1] += 1
        for x in set(echt) & set(v): zaehl[x[0]][0] += 1
        if echt:
            pii += 1
            abgedeckt = set(i for _, s, e in v for i in range(s, e))
            # Satz-Leck (wie in den bisherigen Berichten): irgendein sichtbares Zeichen irgendeiner Stelle
            lecks += any(z["text"][i].isalnum() and i not in abgedeckt for _, s, e in echt for i in range(s, e))
            # Entitäts-Leck (streng, Zeichenebene): eine Stelle gilt als geleakt, sobald EIN Buchstabe/eine Ziffer
            # sichtbar bleibt – ein halb geschwärzter Name zählt also als Leck
            for l, s, e in echt:
                ent[l][0] += 1; ent[l][1] += any(z["text"][i].isalnum() and i not in abgedeckt for i in range(s, e))
        else:
            neg += 1; fa += bool(v)
    tab = []
    for l in sorted(zaehl):
        r, p, g = zaehl[l]; pr = r / p if p else 0; re_ = r / g if g else 0
        tab.append({"Label": l, "Präzision": round(pr, 3), "Trefferquote": round(re_, 3),
                    "F1": round(2 * pr * re_ / (pr + re_), 3) if pr + re_ else 0, "Anzahl echt": g,
                    "Leck-Quote %": round(100 * ent[l][1] / ent[l][0], 1) if ent[l][0] else None})
    stellen = sum(v[0] for v in ent.values()); ent_lecks = sum(v[1] for v in ent.values())
    return {"lecks": lecks, "pii_saetze": pii, "fa": fa, "harmlos": neg, "stellen": stellen, "stellen_geleakt": ent_lecks,
            "Lecks je 1000": round(1000 * lecks / max(pii, 1), 1),
            "Entitäts-Lecks je 1000": round(1000 * ent_lecks / max(stellen, 1), 1),
            "Fehlalarm %": round(100 * fa / max(neg, 1), 1)}, pd.DataFrame(tab)

def asr_artig(z):
    return z.get("stil", "text") != "text"

def nach_stil(saetze, vorh):
    """Dieselbe Bewertung getrennt nach Stil: ASR-artig (Spracheingabe) gegen geschriebenen Text."""
    aus = {}
    for name_, f in (("ASR-artig", asr_artig), ("Text", lambda z: not asr_artig(z))):
        idx = [i for i, z in enumerate(saetze) if f(z)]
        if idx: aus[name_] = bewerten([saetze[i] for i in idx], [vorh[i] for i in idx])[0]
    return aus

def pruefen(model, tok, leer, tau=0.5, temp=1.0, titel="", ausgabe=True):
    lg = logits_fuer(model, tok, val, leer)
    vorh = [spannen(o, l, tau, temp) for o, l in lg]
    erg = {}
    for q in sorted(set(z["quelle"] for z in val)):
        idx = [i for i, z in enumerate(val) if z["quelle"] == q]
        erg[q] = bewerten([val[i] for i in idx], [vorh[i] for i in idx])
        if ausgabe:
            g = erg[q][0]; print(f"{titel} | {q:9s} | Entitäts-Lecks/1000 {g['Entitäts-Lecks je 1000']:6.1f} | Satz-Lecks/1000 {g['Lecks je 1000']:6.1f} | Fehlalarm {g['Fehlalarm %']:5.1f} %")
    return erg, lg

def bewertungszahl(erg):
    # kleiner ist besser: Entitäts-Lecks/1000 durch 10 plus doppelt gewichtete Fehlalarm-%, gemittelt über alle Quellen
    return sum(g["Entitäts-Lecks je 1000"] / 10 + 2 * g["Fehlalarm %"] for g, _ in erg.values()) / len(erg)

def kalibrieren(lg):
    """Temperatur-Kalibrierung (Guo u. a. 2017) auf den Validierungs-Wortteilen, dann τ wählen."""
    X, Y = [], []
    for z, (offs, l) in zip(val, lg):
        pseudo = kodieren_cache.get(id(z))
        if pseudo is None: continue
        n = min(len(pseudo), len(l))
        for i in range(n):
            if pseudo[i] != -100: X.append(l[i]); Y.append(pseudo[i])
    X = torch.tensor(np.array(X)); Y = torch.tensor(Y)
    logt = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([logt], lr=0.1, max_iter=100)
    def schritt():
        opt.zero_grad(); v = F.cross_entropy(X / logt.exp(), Y); v.backward(); return v
    opt.step(schritt)
    temp = float(logt.detach().exp())
    # τ: möglichst wenige Lecks, solange Fehlalarme (über alle Quellen) <= ZIEL_FEHLALARM
    tabelle = []
    for tau in [round(x, 2) for x in np.arange(0.05, 0.96, 0.05)]:
        vorh = [spannen(o, l, tau, temp) for o, l in lg]
        g, _ = bewerten(val, vorh)
        tabelle.append({"tau": tau, "Entitäts-Lecks je 1000": g["Entitäts-Lecks je 1000"], "Lecks je 1000": g["Lecks je 1000"], "Fehlalarm %": g["Fehlalarm %"]})
    df = pd.DataFrame(tabelle)
    ok = df[df["Fehlalarm %"] <= 100 * ZIEL_FEHLALARM]
    if len(ok):
        wahl = ok.sort_values(["Entitäts-Lecks je 1000", "Lecks je 1000", "tau"]).iloc[0]
    else:  # Ziel nicht erreichbar: bester Kompromiss wie bei der Wahl des Zwischenstands
        df["_k"] = df["Entitäts-Lecks je 1000"] / 10 + 2 * df["Fehlalarm %"]
        wahl = df.sort_values(["_k", "tau"]).iloc[0]; df = df.drop(columns="_k")
        print("Hinweis: Fehlalarm-Ziel auf der Validierung nicht erreichbar – Kompromiss-Schwelle gewählt.")
    return temp, float(wahl["tau"]), df

## 7. Training (beide Modelle nacheinander)

Für jedes Modell: laden → trainieren (Zwischenstand nach jeder Epoche) → besten Stand wählen →
Temperatur und Schwelle τ bestimmen. Schritte mit ungültigen Zahlen (NaN) werden übersprungen und gezählt.

In [11]:
import time, math, copy
from transformers import AutoTokenizer, AutoModelForTokenClassification, get_linear_schedule_with_warmup

def lade_modell(pfad, attn="sdpa"):
    kw = dict(num_labels=len(TAGS), id2label=dict(enumerate(TAGS)), label2id=T2I)
    try:
        return AutoModelForTokenClassification.from_pretrained(pfad, attn_implementation=attn, **kw)
    except (ValueError, TypeError, ImportError):
        return AutoModelForTokenClassification.from_pretrained(pfad, **kw)

ERGEBNISSE = {}
for M in MODELLE:
    name, leer = M["name"], M["leerzeichen_vorne"]
    print("\n" + "=" * 70 + f"\nModell: {name} ({M['basis']})\n" + "=" * 70)
    torch.manual_seed(SAAT); random.seed(SAAT)
    tok = AutoTokenizer.from_pretrained(M["basis"])
    train_k = [kodieren(tok, z, leer) for z in train]
    kodieren_cache = {id(z): kodieren(tok, z, leer)["labels"] for z in val}
    model = lade_modell(M["basis"]).to(GERAET)
    print("Parameter:", round(sum(p.numel() for p in model.parameters()) / 1e6, 1), "Mio. | mittlere Länge:",
          round(sum(len(k["input_ids"]) for k in train_k) / len(train_k), 1), "Wortteile")
    gewicht = torch.ones(len(TAGS)); gewicht[0] = GEWICHT_O; gewicht = gewicht.to(GERAET)
    # Getrennte Lernraten: neuer Kopf (classifier/head) schneller, vortrainierter Encoder vorsichtiger
    kopf = [p for n, p in model.named_parameters() if n.startswith(("classifier", "head", "pre_classifier"))]
    kopf_ids = set(id(p) for p in kopf)
    rumpf = [p for p in model.parameters() if id(p) not in kopf_ids]
    print("Kopf-Parameter:", sum(p.numel() for p in kopf), "| Lernraten Encoder/Kopf:", LERNRATE_ENCODER, "/", LERNRATE_KOPF)
    opt = torch.optim.AdamW([{"params": rumpf, "lr": LERNRATE_ENCODER}, {"params": kopf, "lr": LERNRATE_KOPF}], weight_decay=0.01)
    je_epoche = math.ceil(len(train_k) / BATCH)
    gesamt = MAX_SCHRITTE or je_epoche * EPOCHEN
    plan = get_linear_schedule_with_warmup(opt, int(WARMUP * gesamt), gesamt)
    skalierer = torch.amp.GradScaler("cuda", enabled=(GENAUIGKEIT == "fp16"))
    pruef_alle = max(1, je_epoche // PRUEFUNGEN_JE_EPOCHE)
    ordner = f"ausgabe/{name}"; os.makedirs(ordner, exist_ok=True)
    start = time.time(); schritt = 0; nan_schritte = 0; bester = None; verlauf = []; fertig = False

    def pruefpunkt(epoche, nr, summe):
        global bester
        erg, _ = pruefen(model, tok, leer, titel=f"{name} E{epoche} S{nr}")
        wert = bewertungszahl(erg)
        verlauf.append({"epoche": epoche, "schritt": schritt, "minuten": round((time.time() - start) / 60, 1),
                        "fehler": round(summe / max(nr, 1), 4), "bewertung": round(wert, 2),
                        "validierung": {q: g for q, (g, _) in erg.items()}})
        if bester is None or wert < bester[1]:
            model.save_pretrained(f"{ordner}/bester"); tok.save_pretrained(f"{ordner}/bester")
            bester = (f"{ordner}/bester", wert, epoche, schritt)
            print(f"   → neuer bester Stand (Bewertung {wert:.2f})")
        model.train()

    for epoche in range(1, EPOCHEN + 1):
        model.train(); summe = 0.0
        idx = sorted(range(len(train_k)), key=lambda i: len(train_k[i]["input_ids"]) + random.random() * 4)
        stapel_liste = [idx[i:i + BATCH] for i in range(0, len(idx), BATCH)]; random.shuffle(stapel_liste)
        for nr, b in enumerate(stapel_liste, 1):
            ids, att, lab = stapel(tok, [train_k[i] for i in b])
            ids, att, lab = ids.to(GERAET), att.to(GERAET), lab.to(GERAET)
            with torch.autocast(GERAET, dtype=AUTOCAST, enabled=AUTOCAST is not None and GERAET == "cuda"):
                lg = model(input_ids=ids, attention_mask=att).logits
            loss = F.cross_entropy(lg.float().view(-1, len(TAGS)), lab.view(-1), weight=gewicht, ignore_index=-100)
            # (Destillation mit KD_ALPHA/KD_TEMPERATUR käme hier hinzu, wenn das Paket weiche Lehrer-Labels hätte.)
            opt.zero_grad(set_to_none=True)
            if not torch.isfinite(loss):
                nan_schritte += 1; plan.step(); schritt += 1; continue
            skalierer.scale(loss).backward()
            skalierer.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            skalierer.step(opt); skalierer.update(); plan.step()
            schritt += 1; summe += loss.item()
            if nr % 200 == 0:
                print(f"{name} | Epoche {epoche} | Schritt {nr}/{je_epoche} | Fehler {summe / nr:.4f} | NaN {nan_schritte} | {(time.time() - start) / 60:.1f} min")
            if nr % pruef_alle == 0 and nr < len(stapel_liste) - pruef_alle // 3:
                pruefpunkt(epoche, nr, summe)
            if (MAX_SCHRITTE and schritt >= MAX_SCHRITTE) or (time.time() - start) / 60 > M["zeitlimit_min"]:
                if not MAX_SCHRITTE: print("Zeitlimit erreicht – höre sauber auf.")
                fertig = True; break
        model.save_pretrained(f"{ordner}/epoche_{epoche}"); tok.save_pretrained(f"{ordner}/epoche_{epoche}")
        pruefpunkt(epoche, nr, summe)
        if IN_DRIVE_SPEICHERN and IN_COLAB:
            shutil.copytree(f"{ordner}/epoche_{epoche}", f"{DRIVE_ORDNER}/{name}/epoche_{epoche}", dirs_exist_ok=True)
        if fertig:
            break
    minuten = round((time.time() - start) / 60, 1)
    print(f"{name}: fertig nach {minuten} min, {schritt} Schritte, NaN-Schritte: {nan_schritte}, bester Stand: Epoche {bester[2]}, Schritt {bester[3]}")

    # Besten Stand laden, kalibrieren, Schwelle wählen
    model = AutoModelForTokenClassification.from_pretrained(bester[0]).to(GERAET)
    _, lg = pruefen(model, tok, leer, ausgabe=False)
    temp, tau, tau_tab = kalibrieren(lg)
    print(f"Temperatur T = {temp:.3f} | gewählte Schwelle τ = {tau}")
    erg, lg = pruefen(model, tok, leer, tau=tau, temp=temp, titel=f"{name} final")
    stil_erg = nach_stil(val, [spannen(o, l, tau, temp) for o, l in lg])
    for st, g in stil_erg.items():
        print(f"{name} final | Stil {st:9s} | Entitäts-Lecks/1000 {g['Entitäts-Lecks je 1000']:6.1f} | Satz-Lecks/1000 {g['Lecks je 1000']:6.1f} | Fehlalarm {g['Fehlalarm %']:5.1f} % | Stellen {g['stellen']}")
    ERGEBNISSE[name] = {"nach_stil": stil_erg, "modell": M, "bester": bester, "temperatur": temp, "tau": tau, "tau_tabelle": tau_tab,
                        "validierung": erg, "verlauf": verlauf, "minuten": minuten, "schritte": schritt,
                        "nan_schritte": nan_schritte}
    del model, opt; torch.cuda.empty_cache() if GERAET == "cuda" else None


Modell: mmbert-small (jhu-clsp/mmBERT-small)


config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/46.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

distilbert-de E3 S3764 | baukasten | Entitäts-Lecks/1000   39.4 | Satz-Lecks/1000   46.0 | Fehlalarm   7.8 %
distilbert-de E3 S3764 | dev       | Entitäts-Lecks/1000   61.9 | Satz-Lecks/1000   80.0 | Fehlalarm   3.4 %
distilbert-de E3 S3764 | germeval  | Entitäts-Lecks/1000   97.5 | Satz-Lecks/1000  119.3 | Fehlalarm   3.6 %
distilbert-de E3 S3764 | kiji      | Entitäts-Lecks/1000   37.6 | Satz-Lecks/1000   55.8 | Fehlalarm  18.1 %
distilbert-de E3 S3764 | nobody    | Entitäts-Lecks/1000    0.0 | Satz-Lecks/1000    0.0 | Fehlalarm   3.0 %
distilbert-de E3 S3764 | openpii   | Entitäts-Lecks/1000   22.5 | Satz-Lecks/1000   32.9 | Fehlalarm   2.3 %


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

   → neuer bester Stand (Bewertung 17.05)
distilbert-de: fertig nach 11.3 min, 11292 Schritte, NaN-Schritte: 0, bester Stand: Epoche 3, Schritt 11292


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Temperatur T = 1.060 | gewählte Schwelle τ = 0.6
distilbert-de final | baukasten | Entitäts-Lecks/1000   46.9 | Satz-Lecks/1000   54.4 | Fehlalarm   6.2 %
distilbert-de final | dev       | Entitäts-Lecks/1000   61.9 | Satz-Lecks/1000   80.0 | Fehlalarm   2.8 %
distilbert-de final | germeval  | Entitäts-Lecks/1000  111.9 | Satz-Lecks/1000  137.6 | Fehlalarm   2.8 %
distilbert-de final | kiji      | Entitäts-Lecks/1000   37.6 | Satz-Lecks/1000   55.8 | Fehlalarm  17.0 %
distilbert-de final | nobody    | Entitäts-Lecks/1000    0.0 | Satz-Lecks/1000    0.0 | Fehlalarm   3.0 %
distilbert-de final | openpii   | Entitäts-Lecks/1000   25.1 | Satz-Lecks/1000   37.3 | Fehlalarm   2.3 %
distilbert-de final | Stil ASR-artig | Entitäts-Lecks/1000   46.5 | Satz-Lecks/1000   58.8 | Fehlalarm   4.9 % | Stellen 3611
distilbert-de final | Stil Text      | Entitäts-Lecks/1000   33.4 | Satz-Lecks/1000   43.9 | Fehlalarm   4.9 % | Stellen 2367


## 8. Ergebnis der Validierung (beste Stände, mit gewählter Schwelle)

In [12]:
zeilen_ = []
for name, E in ERGEBNISSE.items():
    for q, (g, tab) in E["validierung"].items():
        zeilen_.append({"Modell": name, "Quelle": q, "Entitäts-Lecks je 1000": g["Entitäts-Lecks je 1000"], "Lecks je 1000": g["Lecks je 1000"],
                        "Fehlalarm %": g["Fehlalarm %"], "Stellen": g["stellen"], "Sätze mit PII": g["pii_saetze"], "harmlose Sätze": g["harmlos"]})
    for st, g in E["nach_stil"].items():
        zeilen_.append({"Modell": name, "Quelle": "Stil " + st, "Entitäts-Lecks je 1000": g["Entitäts-Lecks je 1000"], "Lecks je 1000": g["Lecks je 1000"],
                        "Fehlalarm %": g["Fehlalarm %"], "Stellen": g["stellen"], "Sätze mit PII": g["pii_saetze"], "harmlose Sätze": g["harmlos"]})
print(pd.DataFrame(zeilen_).to_string(index=False))
for name, E in ERGEBNISSE.items():
    print(f"\n=== {name}: τ = {E['tau']}, T = {E['temperatur']:.3f} ===")
    print(E["tau_tabelle"].to_string(index=False))
    print(f"\n--- {name}: Labels auf der Validierung 'baukasten' (fremde Vorlagen-Familien) ---")
    if "baukasten" in E["validierung"]:
        print(E["validierung"]["baukasten"][1].to_string(index=False))

       Modell         Quelle  Entitäts-Lecks je 1000  Lecks je 1000  Fehlalarm %  Stellen  Sätze mit PII  harmlose Sätze
 mmbert-small      baukasten                    24.0           28.0          6.0     2922           2500            1031
 mmbert-small            dev                    44.7           57.8          5.0      291            225             179
 mmbert-small       germeval                    72.2           78.0          2.6      277            218             782
 mmbert-small           kiji                    26.1           38.1         14.9      612            394              94
 mmbert-small         nobody                     0.0            0.0          3.7      364            279             164
 mmbert-small        openpii                    18.5           27.4          1.1     1512            912              88
 mmbert-small Stil ASR-artig                    25.5           31.6          4.7     3611           2753            1437
 mmbert-small      Stil Text    

## 9. Für den Filter umwandeln (ONNX, 8 Bit) und prüfen

* Export nach ONNX und Gleichheitsprüfung Torch ↔ ONNX bei 10 Längen (Toleranz 0,001).
* `model_w8.onnx`: 8 Bit für alle Gewichte, **inklusive Embedding** (klein genug für den Raspberry Pi).
* `model_int8.onnx`: klassisches int8 als Vergleich (kann ungenauer sein).
* Für beide Varianten: Lecks und Fehlalarme auf der Validierung.

In [ ]:
import onnx, onnxruntime as ort, zipfile, datetime, logging
logging.getLogger("onnxruntime").setLevel(logging.WARNING)
logging.getLogger("onnxruntime.quantization.matmul_nbits_quantizer").setLevel(logging.WARNING)
for _n in ("onnxscript", "onnx_ir", "torch.onnx"): logging.getLogger(_n).setLevel(logging.WARNING)
import warnings; warnings.filterwarnings("ignore", message=".*axis name.*"); warnings.filterwarnings("ignore", message=".*torchvision.*")
from onnxruntime.quantization.matmul_nbits_quantizer import MatMulNBitsQuantizer, DefaultWeightOnlyQuantConfig
from onnxruntime.quantization import quantize_dynamic, QuantType

class NurLogits(torch.nn.Module):
    def __init__(self, m):
        super().__init__(); self.m = m
    def forward(self, input_ids, attention_mask):
        return self.m(input_ids=input_ids, attention_mask=attention_mask).logits

LAENGEN = [8, 16, 32, 63, 64, 65, 100, 127, 128, 256]
ZIPS = []
for name, E in ERGEBNISSE.items():
    M = E["modell"]; leer = M["leerzeichen_vorne"]; ex = f"export/{name}"
    shutil.rmtree(ex, ignore_errors=True); os.makedirs(ex)
    tok = AutoTokenizer.from_pretrained(E["bester"][0])
    em = AutoModelForTokenClassification.from_pretrained(E["bester"][0], attn_implementation="eager").float().eval().cpu()
    bsp = tok(["Hallo, ich bin Max Muster und wohne in der Hauptstraße 5 in Köln. " * 3], return_tensors="pt")
    arg = dict(input_names=["input_ids", "attention_mask"], output_names=["logits"], opset_version=17,
               dynamic_axes={"input_ids": {0: "b", 1: "n"}, "attention_mask": {0: "b", 1: "n"}, "logits": {0: "b", 1: "n"}})
    def abweichung(pfad):
        s32 = ort.InferenceSession(pfad, providers=["CPUExecutionProvider"])
        langtext = " ".join(z["text"] for z in val[:60]); a = {}
        for L in LAENGEN:
            enc = tok(langtext, truncation=True, max_length=L, return_tensors="pt")
            with torch.no_grad():
                ref = em(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"]).logits.numpy()
            out = s32.run(None, {"input_ids": enc["input_ids"].numpy().astype("int64"), "attention_mask": enc["attention_mask"].numpy().astype("int64")})[0]
            a[L] = float(np.abs(out - ref).max())
        return a
    # Zuerst der neue Exporter (torch.export); der alte TorchScript-Exporter liefert bei DistilBERT falsche Werte.
    abw, exporter = None, None
    try:
        B_, N_ = torch.export.Dim("b"), torch.export.Dim("n", max=8192)
        torch.onnx.export(NurLogits(em).eval(), (bsp["input_ids"], bsp["attention_mask"]), f"{ex}/model.onnx", dynamo=True,
                          input_names=["input_ids", "attention_mask"], output_names=["logits"], opset_version=18,
                          dynamic_shapes={"input_ids": {0: B_, 1: N_}, "attention_mask": {0: B_, 1: N_}}, external_data=False)
        abw, exporter = abweichung(f"{ex}/model.onnx"), "dynamo"
        print(f"{name}: neuer Exporter, größte Abweichung {max(abw.values()):.1e}")
    except Exception as fehler:
        print("Neuer Exporter fehlgeschlagen:", repr(fehler)[:300])
    if abw is None or max(abw.values()) > 1e-3:
        print("Versuche den alten Exporter …")
        torch.onnx.export(NurLogits(em).eval(), (bsp["input_ids"], bsp["attention_mask"]), f"{ex}/model_alt.onnx", dynamo=False, **arg)
        abw_alt = abweichung(f"{ex}/model_alt.onnx")
        if abw is None or max(abw_alt.values()) < max(abw.values()):
            os.replace(f"{ex}/model_alt.onnx", f"{ex}/model.onnx"); abw, exporter = abw_alt, "torchscript"
        else:
            os.remove(f"{ex}/model_alt.onnx")
    gleich_ok = all(v <= 1e-3 for v in abw.values())
    print(f"{name}: Exporter {exporter}, größte Abweichung Torch↔ONNX je Länge:", {k: f"{v:.1e}" for k, v in abw.items()}, "→", "OK" if gleich_ok else "WARNUNG: zu groß!")
    # 8 Bit (Gewichte + Embedding)
    # Schritt 1: Embedding (Gather) auf int8. Schritt 2: alle MatMul-Gewichte 8 Bit (MatMulNBits; bei GLiNER verlustfrei).
    # (Der MatMulNBits-Quantisierer kann Gather nur mit 4 Bit – darum das Embedding getrennt mit 8 Bit.)
    quantize_dynamic(f"{ex}/model.onnx", f"{ex}/model_g8.onnx", weight_type=QuantType.QUInt8, op_types_to_quantize=["Gather"])
    cfg = DefaultWeightOnlyQuantConfig(block_size=32, is_symmetric=True, bits=8, accuracy_level=4)
    q = MatMulNBitsQuantizer(onnx.load(f"{ex}/model_g8.onnx"), algo_config=cfg); q.process()
    os.remove(f"{ex}/model_g8.onnx")
    q.model.save_model_to_file(f"{ex}/model_w8.onnx", use_external_data_format=False)
    quantize_dynamic(f"{ex}/model.onnx", f"{ex}/model_int8.onnx", weight_type=QuantType.QInt8, per_channel=True)
    # Validierung mit den ONNX-Varianten (gleiche Schwelle)
    pruef = {}
    val_onnx = random.Random(0).sample(val, min(len(val), ONNX_PRUEF_SAETZE))  # Stichprobe, damit es auf der Colab-CPU schnell geht
    for var in ["model.onnx", "model_w8.onnx", "model_int8.onnx"]:
        s = ort.InferenceSession(f"{ex}/{var}", providers=["CPUExecutionProvider"])
        vorh = []
        for z in val_onnx:
            enc = zerlegen(tok, [z["text"]], leer, return_tensors="np")
            lo = s.run(None, {"input_ids": enc["input_ids"].astype("int64"), "attention_mask": enc["attention_mask"].astype("int64")})[0][0]
            vorh.append(spannen(enc["offset_mapping"][0], lo, E["tau"], E["temperatur"]))
        g, _ = bewerten(val_onnx, vorh)
        pruef[var] = {"MB": round(os.path.getsize(f"{ex}/{var}") / 1e6), "Entitäts-Lecks je 1000": g["Entitäts-Lecks je 1000"],
                      "Lecks je 1000": g["Lecks je 1000"], "Fehlalarm %": g["Fehlalarm %"]}
        print(f"   {var:16s} {pruef[var]}")
    # fp32 nur mitliefern, wenn beide 8-Bit-Fassungen spürbar schlechter sind (Lecks +10 % und +2 je 1000, oder Fehlalarm +1,5 Punkte)
    def schlechter(v):
        r, q = pruef["model.onnx"], pruef[v]
        return q["Entitäts-Lecks je 1000"] > r["Entitäts-Lecks je 1000"] * 1.1 + 2 or q["Fehlalarm %"] > r["Fehlalarm %"] + 1.5
    fp32_dabei = schlechter("model_w8.onnx") and schlechter("model_int8.onnx")
    if fp32_dabei:
        print("   Hinweis: 8-Bit verliert spürbar – model.onnx (fp32) kommt mit ins Zip.")
    else:
        os.remove(f"{ex}/model.onnx")
    for f in os.listdir(ex):
        if f.endswith(".data"): os.remove(os.path.join(ex, f))
    # Zip packen
    tok.save_pretrained(ex)
    json.dump({"tags": TAGS, "labels": LABELS, "hilfslabels": sorted(HILFS), "temperatur": E["temperatur"], "tau": E["tau"],
               "leerzeichen_vorne": leer, "max_laenge": MAX_LAENGE, "art9_zusammengelegt": meta.get("art9_zusammengelegt", []),
               "hinweis": "Schwärzen, wenn Summe P(schwärzbare Klassen) nach Temperatur >= tau. ORT/ORG nie schwärzen."},
              open(f"{ex}/labels.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    json.dump({"name": name, "basismodell": M["basis"], "epochen": EPOCHEN, "lernrate_encoder": LERNRATE_ENCODER, "lernrate_kopf": LERNRATE_KOPF, "batch": BATCH, "warmup": WARMUP,
               "gewicht_o": GEWICHT_O, "ziel_fehlalarm": ZIEL_FEHLALARM, "max_laenge": MAX_LAENGE, "trainingssaetze": len(train),
               "minuten": E["minuten"], "schritte": E["schritte"], "nan_schritte": E["nan_schritte"],
               "bester_stand": {"epoche": E["bester"][2], "schritt": E["bester"][3]}, "verlauf": E["verlauf"],
               "tau_tabelle": E["tau_tabelle"].to_dict("records"), "onnx_abweichung": abw, "onnx_gleich_ok": gleich_ok, "onnx_exporter": exporter, "fp32_im_zip": fp32_dabei,
               "onnx_pruefung": pruef, "validierung_nach_stil": E["nach_stil"],
               "validierung": {q: g for q, (g, _) in E["validierung"].items()}, "probelauf": PROBE, "genauigkeit": GENAUIGKEIT, "destillation": MIT_WEICH,
               "geraet": torch.cuda.get_device_name(0) if GERAET == "cuda" else "cpu",
               "versionen": {"torch": torch.__version__, "transformers": transformers.__version__, "onnx": onnx.__version__,
                             "onnxruntime": ort.__version__}},
              open(f"{ex}/lauf_info.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    if "lizenzen" in meta:
        open(f"{ex}/LIZENZEN.md", "w", encoding="utf-8").write(meta["lizenzen"])
    zn = f"pii-modell-{name}-" + datetime.datetime.now().strftime("%Y%m%d-%H%M") + ("-probe" if PROBE else "") + ".zip"
    with zipfile.ZipFile(zn, "w", zipfile.ZIP_DEFLATED) as z:
        for f in sorted(os.listdir(ex)):
            z.write(os.path.join(ex, f), f)
    ZIPS.append(zn)
    print(f"Zip: {zn} ({round(os.path.getsize(zn) / 1e6)} MB)")

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

[torch.onnx] Obtain model graph for `NurLogits([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `NurLogits([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
mmbert-small: neuer Exporter, größte Abweichung 3.3e-04
mmbert-small: Exporter dynamo, größte Abweichung Torch↔ONNX je Länge: {8: '1.5e-05', 16: '2.0e-05', 32: '1.4e-04', 63: '1.3e-04', 64: '3.3e-04', 65: '2.0e-04', 100: '5.1e-05', 127: '1.1e-04', 128: '1.2e-04', 256: '1.1e-04'} → OK


   model.onnx       {'MB': 565, 'Entitäts-Lecks je 1000': 16.4, 'Lecks je 1000': 21.8, 'Fehlalarm %': 5.1}


## 10. Herunterladen

In [ ]:
for zn in ZIPS:
    if IN_DRIVE_SPEICHERN and IN_COLAB:
        shutil.copy(zn, DRIVE_ORDNER); print("In Drive gespeichert:", f"{DRIVE_ORDNER}/{zn}")
    if IN_COLAB:
        from google.colab import files
        files.download(zn)
print("Fertig:", ZIPS)
print("Falls kein Download startet: links auf das Ordner-Symbol klicken, Zip-Datei rechts anklicken → Herunterladen.")